# FashionMNIST 动手实战

独立完成一个图像分类项目的完整流程。

| Stage | 内容 | 时间 |
|-------|------|------|
| 1 | 数据加载与可视化 | 3 min |
| 2 | 模型定义 | 10 min |
| 3 | 损失函数 + 优化器 | 2 min |
| 4 | 训练 / 测试函数 | 10 min |
| 5 | 运行训练 + 推理验证 | 5 min |

**参考资料**: 遇到困难时翻阅 `pytorch_cifar100.ipynb` 中的对应章节。

## Stage 1: 数据加载与可视化 (3 min)

运行下面的 cell，熟悉 FashionMNIST 数据集的基本信息。

In [5]:
import torch
from torch import nn
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# device = "cuda" if torch.cuda.is_available() else "cpu"
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

Device: cpu


In [6]:
training_data = datasets.FashionMNIST(root="data", train=True, download=True, transform=ToTensor())
test_data = datasets.FashionMNIST(root="data", train=False, download=True, transform=ToTensor())

batch_size = 64
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

for X, y in test_dataloader:
    print(f"输入 X: {X.shape}  (B, C, H, W)")
    print(f"标签 y: {y.shape}")
    print(f"通道数: {X.shape[1]}, 图像尺寸: {X.shape[2]}x{X.shape[3]}, 类别数: 10")
    break

输入 X: torch.Size([64, 1, 28, 28])  (B, C, H, W)
标签 y: torch.Size([64])
通道数: 1, 图像尺寸: 28x28, 类别数: 10


In [ ]:
classes = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

images, labels = next(iter(train_dataloader))
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(classes[labels[i]], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Stage 2: 模型定义 (10 min)

设计一个 CNN 对 FashionMNIST 做 10 分类。

**关键信息**:
- 输入: `[B, 1, 28, 28]` (灰度图, 1 通道)
- 输出: `[B, 10]` (10 个类别的 logits)

**建议结构** (你也可以自己设计):

```
Conv2d(1, 32, 3, padding=1) -> BatchNorm2d(32) -> ReLU -> MaxPool2d(2)   => [B, 32, 14, 14]
Conv2d(32, 64, 3, padding=1) -> BatchNorm2d(64) -> ReLU -> MaxPool2d(2)  => [B, 64, 7, 7]
Flatten                                                                    => [B, 64*7*7]
Linear(64*7*7, 128) -> ReLU -> Linear(128, 10)                            => [B, 10]
```

**参考**: `pytorch_cifar100.ipynb` 的 SimpleCNN (注意 CIFAR 是 3 通道 32x32, 这里是 1 通道 28x28)

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # TODO: 第一个卷积块
            # Conv2d(?, ?, ?, padding=?) -> BatchNorm2d(?) -> ReLU() -> MaxPool2d(?)

            # TODO: 第二个卷积块
            # Conv2d(?, ?, ?, padding=?) -> BatchNorm2d(?) -> ReLU() -> MaxPool2d(?)
        )
        self.classifier = nn.Sequential(
            # TODO: Flatten -> Linear -> ReLU -> Linear
            # 提示: Flatten 后的维度 = 通道数 * 高 * 宽
        )

    def forward(self, x):
        # TODO: features -> classifier
        pass

model = MyModel().to(device)
print(model)

# 验证: 用 dummy input 检查输出维度
dummy = torch.randn(2, 1, 28, 28).to(device)
try:
    out = model(dummy)
    print(f"输出 shape: {out.shape}")  # 应为 [2, 10]
except Exception as e:
    print(f"出错: {e}")

## Stage 3: 损失函数 + 优化器 (2 min)

两行代码。想想：
- 10 分类任务用哪种损失函数？(参考 CIFAR notebook 的损失函数章节)
- 优化器用什么？学习率建议 `1e-3`

In [ ]:
# TODO: 损失函数 (提示: nn.???Loss)
loss_fn = ...

# TODO: 优化器 (提示: torch.optim.???, 别忘了传 model.parameters())
optimizer = ...

## Stage 4: 训练 / 测试函数 (10 min)

这是最核心的部分。下面给出了函数骨架和提示，你需要填入关键的几行代码。

**参考**: `pytorch_cifar100.ipynb` 的训练函数和测试函数章节。

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch_idx, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # TODO: 前向传播 — 将 X 送入 model
        pred = ...

        # TODO: 计算损失 — 用 loss_fn 比较 pred 和 y
        loss = ...

        # TODO: 反向传播三步 (参考 CIFAR notebook)
        # 第 1 步: 清零梯度
        # 第 2 步: 反向传播
        # 第 3 步: 更新参数

        if batch_idx % 200 == 0:
            print(f"  loss: {loss.item():.4f}  [{batch_idx * len(X):>5d}/{len(dataloader.dataset)}]")

In [ ]:
def test(dataloader, model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)

            # TODO: 前向传播
            pred = ...

            # TODO: 计算正确预测数
            # 提示: pred.argmax(1) 取每行最大值的索引
            #        与 y 比较, 统计相等的个数
            correct += ...
            total += y.size(0)

    accuracy = 100 * correct / total
    print(f"  Accuracy: {accuracy:.1f}%")

## Stage 5: 运行训练 + 推理 (5 min)

运行训练，观察 loss 下降和准确率提升。然后在测试样本上验证。

In [ ]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model)
print("训练完成!")

In [ ]:
model.eval()
classes = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for i, ax in enumerate(axes.flat):
    img, label = test_data[i]
    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device))
    pred_label = pred.argmax(1).item()
    color = "green" if pred_label == label else "red"
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"{classes[pred_label]} (gt:{classes[label]})", fontsize=9, color=color)
    ax.axis("off")
plt.suptitle("Green=correct, Red=wrong", fontsize=12)
plt.tight_layout()
plt.show()